In [ ]:
# ----------------------------------------------------------------------
# Experiment 1A: Single Layer Perceptron (SLP) - Breast Cancer Wisconsin
# ----------------------------------------------------------------------

# Step 1: Import Required Libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score,
                             confusion_matrix,
                             classification_report)


# Step 2: Load and Explore the Dataset
# Load dataset
data = load_breast_cancer()
X = data.data      # Shape: (569, 30)
y = data.target    # Shape: (569,)  -- 0: Malignant, 1: Benign

print("Dataset shape    :", X.shape)
print("Number of classes:", len(data.target_names))
print("Class names      :", data.target_names)
print("Feature names    :", data.feature_names[:5], "...")
print("Class distribution:", dict(zip(data.target_names,
       [sum(y==0), sum(y==1)])))


# Step 3: Preprocess the Data
# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Feature scaling (StandardScaler: mean=0, std=1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # Fit on train only
X_test  = scaler.transform(X_test)       # Apply same scale to test

print("Training samples :", X_train.shape)
print("Testing samples  :", X_test.shape)


# Step 4: Define the Perceptron Class
class SingleLayerPerceptron:
    """
    Single Layer Perceptron with Perceptron Learning Rule.
    Supports binary classification (labels 0 and 1).
    """

    def __init__(self, learning_rate=0.01, n_epochs=100):
        self.lr       = learning_rate
        self.n_epochs = n_epochs
        self.weights  = None   # Will be set during fit()
        self.bias     = None

    # ------ Activation Function (Step Function) ------
    def _step(self, z):
        return np.where(z >= 0, 1, 0)

    # ------ Forward Pass ------
    def _predict_raw(self, X):
        """Compute weighted sum: z = X.w + b"""
        return np.dot(X, self.weights) + self.bias

    # ------ Training ------
    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialize weights to zeros
        self.weights = np.zeros(n_features)
        self.bias    = 0

        self.errors_per_epoch = []   # Track misclassifications

        for epoch in range(self.n_epochs):
            errors = 0

            # Iterate over every training sample
            for xi, yi in zip(X, y):
                # Forward pass: compute prediction
                z    = np.dot(xi, self.weights) + self.bias
                y_hat = self._step(z)

                # Compute error
                error = yi - y_hat

                # Weight update (Perceptron Learning Rule)
                self.weights += self.lr * error * xi
                self.bias    += self.lr * error

                # Count misclassifications
                errors += int(error != 0)

            self.errors_per_epoch.append(errors)

            # Early stopping if no errors
            if errors == 0:
                print(f"Converged at epoch {epoch + 1}")
                break

        return self

    # ------ Prediction ------
    def predict(self, X):
        return self._step(self._predict_raw(X))


# Step 5: Train the Perceptron
# Instantiate the perceptron
slp = SingleLayerPerceptron(learning_rate=0.01, n_epochs=100)

# Train on training data
slp.fit(X_train, y_train)

print("Training complete.")
print("Final weight vector (first 5):", slp.weights[:5].round(4))
print("Bias:", round(slp.bias, 4))


# Step 6: Evaluate the Model
# Predict on test data
y_pred = slp.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Detailed Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=["Malignant", "Benign"]))


# Step 7: Plot the Error Curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(slp.errors_per_epoch) + 1),
         slp.errors_per_epoch,
         marker='o', color='steelblue', linewidth=2)

plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Number of Misclassifications", fontsize=12)
plt.title("SLP Learning Curve: Misclassifications per Epoch",
          fontsize=13, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("slp_learning_curve.png", dpi=150)
plt.show()
print("Plot saved as 'slp_learning_curve.png'")


# Step 8: Visualize the Confusion Matrix as a Heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)

classes = ["Malignant", "Benign"]
tick_marks = range(len(classes))
ax.set_xticks(tick_marks); ax.set_xticklabels(classes, fontsize=11)
ax.set_yticks(tick_marks); ax.set_yticklabels(classes, fontsize=11)

# Add text annotations inside cells
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14, fontweight='bold')

ax.set_ylabel('True Label',      fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title('Confusion Matrix -- SLP', fontsize=13,
             fontweight='bold')
plt.tight_layout()
plt.savefig("slp_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# Experiment 1B: Multilayer Perceptron (MLP) - Pima Indians Diabetes
# ----------------------------------------------------------------------

# Step 1: Install Required Libraries
# Run this cell once. Restart the runtime after installation if prompted.
!pip install numpy pandas matplotlib seaborn scikit-learn tensorflow


# Step 2: Import All Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("All libraries imported successfully.")


# Step 3: Load and Preview the Dataset
df = pd.read_csv('diabetes.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())


# Step 4: Explore the Dataset
print("Statistical Summary:")
print(df.describe().round(2))

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nClass distribution (Outcome column):")
print(df['Outcome'].value_counts())


# Step 5: Separate Features and the Target Label
X = df.drop('Outcome', axis=1).values   # Shape: (768, 8)
y = df['Outcome'].values                # Shape: (768,)

print("Feature matrix shape:", X.shape)
print("Label array shape   :", y.shape)
print("\nSample of first 3 feature rows:\n", X[:3])
print("\nFirst 3 labels:", y[:3])


# Step 6: Split into Training and Test Sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% held out for testing
    random_state=42   # Fixed seed for reproducibility
)

print("Training samples:", X_train.shape[0])
print("Testing  samples:", X_test.shape[0])
print("\nTraining label distribution:", dict(zip(*np.unique(y_train, return_counts=True))))


# Step 7: Normalise the Input Features
scaler = StandardScaler()

# Fit on training data ONLY, then transform both sets
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("Mean of training features after scaling (approx. 0):")
print(X_train.mean(axis=0).round(3))

print("\nStd of training features after scaling (approx. 1):")
print(X_train.std(axis=0).round(3))


# Step 8: Build the MLP Architecture
model = Sequential([
    Dense(16, activation='relu', input_shape=(8,)),  # Hidden Layer 1
    Dense(8,  activation='relu'),                    # Hidden Layer 2
    Dense(1,  activation='sigmoid')                  # Output Layer
])

model.summary()


# Step 9: Compile the Model
model.compile(
    optimizer='adam',             # Adaptive learning-rate optimizer
    loss='binary_crossentropy',   # Correct loss for binary (0/1) output
    metrics=['accuracy']          # Metric reported at each epoch
)

print("Model compiled successfully.")


# Step 10: Train the Model
history = model.fit(
    X_train, y_train,
    epochs=100,           # Full passes through training data
    batch_size=16,        # Update weights after every 16 patients
    validation_split=0.2, # Hold out 20% of train data to watch for overfitting
    verbose=1
)

print("\nFinal training accuracy  :", round(history.history['accuracy'][-1], 4))
print("Final validation accuracy:", round(history.history['val_accuracy'][-1], 4))


# Step 11: Visualise the Training Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy plot
axes[0].plot(history.history['accuracy'],
             label='Training', color='steelblue', linewidth=1.5)
axes[0].plot(history.history['val_accuracy'],
             label='Validation', color='darkorange',
             linewidth=1.5, linestyle='--')
axes[0].set_title('Accuracy over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'],
             label='Training', color='steelblue', linewidth=1.5)
axes[1].plot(history.history['val_loss'],
             label='Validation', color='darkorange',
             linewidth=1.5, linestyle='--')
axes[1].set_title('Loss over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Step 12: Evaluate the Model on the Test Set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}  ({test_acc * 100:.1f}%)")
print(f"\nCorrectly classified: {int(test_acc * len(y_test))}"
      f" out of {len(y_test)} patients.")


# Step 13: Classification Report and Confusion Matrix
# Convert predicted probabilities to binary labels
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Detailed per-class metrics
print("Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['No Diabetes (0)', 'Diabetes (1)']))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()


# Step 14: Predict for a New Patient
# Medical details of the new patient
new_patient = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])

# Apply the SAME scaler used during training
new_patient_scaled = scaler.transform(new_patient)

# Get the predicted probability
prob = model.predict(new_patient_scaled, verbose=0)[0][0]
label = 'Diabetes' if prob > 0.5 else 'No Diabetes'

print("Patient Details:")
print(f"  Pregnancies: 6  |  Glucose: 148  |  Blood Pressure: 72")
print(f"  Skin Thickness: 35  |  Insulin: 0  |  BMI: 33.6")
print(f"  Diabetes Pedigree Function: 0.627  |  Age: 50")
print(f"\nPredicted Probability of Diabetes : {prob:.4f}  ({prob*100:.1f}%)")
print(f"Predicted Outcome                 : {label}")

In [ ]:
# ----------------------------------------------------------------------
# Experiment 2: Backpropagation Algorithm - Iris
# ----------------------------------------------------------------------

# Step 1: Install Required Libraries
# Run once; restart runtime if prompted
!pip install numpy pandas matplotlib scikit-learn


# Step 2: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")
print("NumPy version:", np.__version__)


# Step 3: Load and Inspect the Dataset
df = pd.read_csv('iris.csv')

print("Shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

print("\nClass distribution:")
print(df['Species'].value_counts())


# Step 4: Preprocess: Encode Labels and Normalise Features
# Drop the Id column and extract features / labels
df = df.drop('Id', axis=1)
X_raw = df.iloc[:, :-1].values   # All columns except last -> (150, 4)
y_raw  = df.iloc[:, -1].values   # Last column (Species) -> (150,)

# Encode string class names to integers: setosa=0, versicolor=1, virginica=2
le = LabelEncoder()
y_int = le.fit_transform(y_raw)

print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Normalise features to [0, 1] so all inputs are on the same scale
scaler = MinMaxScaler()
X = scaler.fit_transform(X_raw)

print("\nRaw feature sample (row 0):   ", X_raw[0])
print("Scaled feature sample (row 0):", X[0].round(4))


# Step 5: One-Hot Encode Labels and Split the Data
# One-hot encode: integer 0/1/2  ->  3-dimensional binary vector
# np.eye(3) is a 3x3 identity matrix; indexing it with y_int gives one-hot rows
Y = np.eye(3)[y_int]   # Shape: (150, 3)

print("One-hot encoding examples:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls:20s} (int={i})  ->  {Y[y_int == i][0]}")

# Stratified split: 80% train, 20% test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=y_int
)
# Keep integer labels for accuracy evaluation
y_test_int = np.argmax(Y_test, axis=1)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing  samples: {X_test.shape[0]}")


# Step 6: Initialise Network Parameters
np.random.seed(42)

params = {
    'W1': np.random.randn(4, 8) * 0.01,   # (4, 8)  input -> hidden
    'b1': np.zeros((1, 8)),                # (1, 8)  hidden bias
    'W2': np.random.randn(8, 3) * 0.01,   # (8, 3)  hidden -> output
    'b2': np.zeros((1, 3))                 # (1, 3)  output bias
}

total_params = sum(v.size for v in params.values())
print("Parameter shapes:")
for name, val in params.items():
    print(f"  {name}: {val.shape}  ({val.size} values)")
print(f"\nTotal trainable parameters: {total_params}")


# Step 7: Implement Activation Functions and Their Derivatives
def relu(Z):
    """Rectified Linear Unit: max(0, z) applied element-wise."""
    return np.maximum(0, Z)

def relu_deriv(Z):
    """Derivative of ReLU: 1 where Z > 0, else 0."""
    return (Z > 0).astype(float)

def softmax(Z):
    """Row-wise Softmax with numerical stability (subtract row max)."""
    # Subtracting the row maximum prevents overflow in np.exp()
    shift = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(shift)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

# Quick sanity checks
print("relu([-3, 0, 5]) =", relu(np.array([-3, 0, 5])))
print("relu_deriv([-3, 0, 5]) =", relu_deriv(np.array([-3, 0, 5])))

test_logits = np.array([[2.0, 1.0, 0.5]])
probs = softmax(test_logits)
print("softmax([2.0, 1.0, 0.5]) =", probs.round(4))
print("Sum of softmax outputs =", probs.sum().round(6))


# Step 8: Implement Forward Propagation
def forward(X, params):
    """
    Compute the forward pass through the two-layer network.
    Returns predicted probabilities A2 and a cache of intermediates
    needed by the backward pass.
    """
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']

    Z1 = X @ W1 + b1          # (m, 4) x (4, 8) + (1, 8) -> (m, 8)
    A1 = relu(Z1)              # (m, 8)

    Z2 = A1 @ W2 + b2         # (m, 8) x (8, 3) + (1, 3) -> (m, 3)
    A2 = softmax(Z2)           # (m, 3) -- class probabilities

    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}
    return A2, cache

# Test with the initial (near-zero) parameters on the full training set
A2_init, _ = forward(X_train, params)
print("A2 shape:", A2_init.shape)
print("First 3 rows of A2 (initial probabilities):")
print(A2_init[:3].round(4))
print("\nEach row sums to:", A2_init[:3].sum(axis=1).round(6))


# Step 9: Implement the Cross-Entropy Loss Function
def compute_loss(A2, Y):
    """
    Categorical Cross-Entropy loss averaged over all m samples.
    eps prevents log(0) = -infinity.
    """
    m   = Y.shape[0]
    eps = 1e-8
    loss = -np.sum(Y * np.log(A2 + eps)) / m
    return loss

# Verify the initial loss
initial_loss = compute_loss(A2_init, Y_train)
print(f"Initial loss: {initial_loss:.4f}")
print(f"Expected:     {-np.log(1/3):.4f}  (= -log(1/3), random-chance baseline)")


# Step 10: Implement Backward Propagation
def backward(X, Y, params, cache):
    """
    Compute gradients of the loss w.r.t. all parameters using the chain rule.
    Returns a dictionary of gradients matching the shape of params.
    """
    m  = X.shape[0]
    W2 = params['W2']
    A1 = cache['A1']
    A2 = cache['A2']
    Z1 = cache['Z1']

    # 1. Output error: combined CCE + Softmax gradient (averaged over batch)
    dZ2 = (A2 - Y) / m                               # (m, 3)

    # 2. Gradients for W2 and b2
    dW2 = A1.T @ dZ2                                  # (8, m) x (m, 3) -> (8, 3)
    db2 = np.sum(dZ2, axis=0, keepdims=True)           # (1, 3)

    # 3. Propagate error back through W2
    dA1 = dZ2 @ W2.T                                  # (m, 3) x (3, 8) -> (m, 8)

    # 4. Apply ReLU derivative (zero out gradient where Z1 <= 0)
    dZ1 = dA1 * relu_deriv(Z1)                        # (m, 8)

    # 5. Gradients for W1 and b1
    dW1 = X.T @ dZ1                                   # (4, m) x (m, 8) -> (4, 8)
    db1 = np.sum(dZ1, axis=0, keepdims=True)           # (1, 8)

    return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

# Test: run one backward pass and check gradient shapes
A2_test, cache_test = forward(X_train, params)
grads = backward(X_train, Y_train, params, cache_test)
print("Gradient shapes:")
for name, g in grads.items():
    print(f"  d{name[1:]}: {g.shape}  (matches {params[name[1:]].shape})")


# Step 11: Implement the Weight Update
def update(params, grads, lr):
    """
    Apply gradient descent: subtract scaled gradients from each parameter.
    Modifies params in-place and also returns it.
    """
    params['W1'] -= lr * grads['dW1']
    params['b1'] -= lr * grads['db1']
    params['W2'] -= lr * grads['dW2']
    params['b2'] -= lr * grads['db2']
    return params

# Demonstrate one complete iteration (forward -> loss -> backward -> update)
A2_demo, cache_demo = forward(X_train, params)
loss_before = compute_loss(A2_demo, Y_train)

grads_demo  = backward(X_train, Y_train, params, cache_demo)
params      = update(params, grads_demo, lr=0.1)

A2_demo2, _ = forward(X_train, params)
loss_after  = compute_loss(A2_demo2, Y_train)

print(f"Loss before one update: {loss_before:.6f}")
print(f"Loss after  one update: {loss_after:.6f}")
print(f"Loss decreased by:      {loss_before - loss_after:.6f}")


# Step 12: Run the Full Training Loop
# Reset parameters to their original random initialisation
np.random.seed(42)
params = {
    'W1': np.random.randn(4, 8) * 0.01,
    'b1': np.zeros((1, 8)),
    'W2': np.random.randn(8, 3) * 0.01,
    'b2': np.zeros((1, 3))
}

# Hyperparameters
lr     = 0.01
epochs = 1000

loss_history = []

for epoch in range(epochs):
    # Forward pass
    A2, cache = forward(X_train, params)

    # Compute loss
    loss = compute_loss(A2, Y_train)
    loss_history.append(loss)

    # Backward pass
    grads = backward(X_train, Y_train, params, cache)

    # Update weights
    params = update(params, grads, lr)

    # Print progress every 100 epochs
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:4d}  |  Loss: {loss:.4f}")

print(f"\nInitial loss : {loss_history[0]:.4f}")
print(f"Final loss   : {loss_history[-1]:.4f}")


# Step 13: Plot the Training Loss Curve and Evaluate Accuracy
# --- Loss curve ---
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='steelblue', linewidth=1.5)
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Cross-Entropy Loss', fontsize=11)
plt.title('Training Loss over 1000 Epochs', fontweight='bold', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Training accuracy ---
A2_train, _ = forward(X_train, params)
y_pred_train = np.argmax(A2_train, axis=1)
y_true_train = np.argmax(Y_train,  axis=1)
train_acc = np.mean(y_pred_train == y_true_train)

# --- Test accuracy ---
A2_test, _ = forward(X_test, params)
y_pred_test = np.argmax(A2_test, axis=1)
test_acc    = np.mean(y_pred_test == y_test_int)

print(f"Training Accuracy: {train_acc:.4f}  ({train_acc*100:.1f}%)")
print(f"Test Accuracy    : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print(f"\nCorrectly classified: {int(test_acc * len(y_test_int))}"
      f" out of {len(y_test_int)} test flowers.")

# Per-class breakdown
print("\nPer-class predictions on test set:")
for i, cls in enumerate(le.classes_):
    mask = (y_test_int == i)
    cls_acc = np.mean(y_pred_test[mask] == i)
    print(f"  {cls:20s}: {cls_acc*100:.1f}%")


# Step 14: Predict for a New Flower Sample
# A new flower whose species we do not know
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])   # (SepalL, SepalW, PetalL, PetalW)

# Scale using the SAME scaler fitted on the training data
new_scaled = scaler.transform(new_flower)         # (1, 4)

# Forward pass through our trained network
probs, _ = forward(new_scaled, params)            # (1, 3)

pred_int   = np.argmax(probs, axis=1)[0]
pred_label = le.classes_[pred_int]

print("New flower measurements:")
print(f"  Sepal Length: {new_flower[0,0]} cm  |  Sepal Width:{new_flower[0,1]} cm")
print(f"  Petal Length: {new_flower[0,2]} cm  |  Petal Width:{new_flower[0,3]} cm")
print("\nClass probabilities:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls:20s}: {probs[0,i]:.4f}  ({probs[0,i]*100:.1f}%)")
print(f"\nPredicted species: {pred_label}")
print(f"True species:      Iris-setosa  (row 0 of the original CSV)")

In [ ]:
# ----------------------------------------------------------------------
# Experiment 3: Optimization Algorithms (SGD, Momentum, Adam) - MNIST
# ----------------------------------------------------------------------

# Step 1: Install and Verify Libraries
# Run once; TensorFlow already ships with NumPy and Matplotlib in Colab
!pip install tensorflow --quiet

import tensorflow as tf
print("TensorFlow version:", tf.__version__)


# Step 2: Import All Required Modules
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
import tensorflow as tf

print("All modules imported successfully.")


# Step 3: Load the MNIST Dataset
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

print("Training set:")
print(f"  Images : {X_train_raw.shape}  "
      f"(dtype: {X_train_raw.dtype})")
print(f"  Labels : {y_train_raw.shape}  "
      f"(values: {np.unique(y_train_raw)})")

print("\nTest set:")
print(f"  Images : {X_test_raw.shape}")
print(f"  Labels : {y_test_raw.shape}")

print("\nPixel value range:", X_train_raw.min(),
      "to", X_train_raw.max())


# Step 4: Explore and Visualise Sample Images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_raw[i], cmap='gray')
    ax.set_title(f'Label: {y_train_raw[i]}',
                 fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('First 10 MNIST Training Images',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Digit distribution in the training set
unique, counts = np.unique(y_train_raw, return_counts=True)
print("Class distribution in training set:")
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count} images")


# Step 5: Preprocess: Flatten, Normalise, and One-Hot Encode
# 1. Flatten 2D images to 1D vectors and normalise pixel values to [0, 1]
X_train = X_train_raw.reshape(-1, 784).astype('float32') /255.0
X_test  = X_test_raw.reshape(-1, 784).astype('float32') /255.0

# 2. One-hot encode integer labels to 10-dimensional binary vectors
y_train = to_categorical(y_train_raw, num_classes=10)
y_test  = to_categorical(y_test_raw,  num_classes=10)

print("After preprocessing:")
print(f"  X_train : {X_train.shape}  "
      f"pixel range [{X_train.min():.1f}, {X_train.max():.1f}]")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"\nExample -- digit '5' becomes: {y_train[0]}")
print(f"(True label was: {y_train_raw[0]})")


# Step 6: Build the Model Factory Function
def build_model():
    """
    Returns a freshly initialised, uncompiled Dense network.
    Called once per optimizer to ensure a fair comparison:
    all three models start from identical random weights.
    """
    tf.random.set_seed(42)
    model = Sequential([
        Dense(128, activation='relu', input_shape=(784,)),
        Dense(64,  activation='relu'),
        Dense(10,  activation='softmax')
    ])
    return model

# Print the architecture once
build_model().summary()


# Step 7: Configure the Three Optimizers
# Each optimizer is a separate object with its own hyperparameters
opt_sgd = SGD(learning_rate=0.01)
opt_mom = SGD(learning_rate=0.01, momentum=0.9)
opt_adam = Adam(learning_rate=0.001)

print("Optimizer configurations:")
print(f"  Plain SGD    : SGD(lr=0.01)")
print(f"  SGD+Momentum : SGD(lr=0.01, momentum=0.9)")
print(f"  Adam         : Adam(lr=0.001)")
print()
print("Key difference: Adam uses a LOWER learning rate (0.001 vs 0.01)")
print("because its adaptive scaling amplifies effective steps per parameter.")


# Step 8: Train with Plain SGD
print("=" * 50)
print("Training Model 1: Plain SGD")
print("=" * 50)

model_sgd = build_model()
model_sgd.compile(
    optimizer=opt_sgd,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist_sgd = model_sgd.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print(f"\nFinal training accuracy  : "
      f"{hist_sgd.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: "
      f"{hist_sgd.history['val_accuracy'][-1]:.4f}")


# Step 9: Train with SGD + Momentum
print("=" * 50)
print("Training Model 2: SGD with Momentum")
print("=" * 50)

model_mom = build_model()   # Fresh model -- same initial weights
model_mom.compile(
    optimizer=opt_mom,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist_mom = model_mom.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print(f"\nFinal training accuracy  : "
      f"{hist_mom.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: "
      f"{hist_mom.history['val_accuracy'][-1]:.4f}")


# Step 10: Train with Adam
print("=" * 50)
print("Training Model 3: Adam")
print("=" * 50)

model_adam = build_model()   # Fresh model again
model_adam.compile(
    optimizer=opt_adam,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

hist_adam = model_adam.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print(f"\nFinal training accuracy  : "
      f"{hist_adam.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: "
      f"{hist_adam.history['val_accuracy'][-1]:.4f}")


# Step 11: Plot Comparative Training Curves
epochs_range = range(1, 16)

labels   = ['Plain SGD', 'SGD + Momentum', 'Adam']
colours  = ['tomato', 'steelblue', 'forestgreen']
histories = [hist_sgd, hist_mom, hist_adam]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Training Loss ---
for hist, label, col in zip(histories, labels, colours):
    axes[0].plot(epochs_range, hist.history['loss'],
                 label=label, color=col, linewidth=2)
axes[0].set_title('Training Loss per Epoch',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Categorical Cross-Entropy Loss', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(epochs_range)

# --- Validation Accuracy ---
for hist, label, col in zip(histories, labels, colours):
    axes[1].plot(epochs_range, hist.history['val_accuracy'],
                 label=label, color=col, linewidth=2)
axes[1].axhline(y=0.95, color='gray', linestyle='--',
                linewidth=1, label='95% target')
axes[1].set_title('Validation Accuracy per Epoch',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Validation Accuracy', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(epochs_range)

plt.tight_layout()
plt.savefig('optimizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as 'optimizer_comparison.png'")


# Step 12: Visualise Predictions from the Best Model
# Use the Adam model (best performer) to predict on the first 25 test images
y_pred_prob = model_adam.predict(X_test[:25], verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = np.argmax(y_test[:25], axis=1)

fig, axes = plt.subplots(5, 5, figsize=(9, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_raw[i], cmap='gray')
    correct = (y_pred[i] == y_true[i])
    colour  = 'green' if correct else 'red'
    ax.set_title(f'P:{y_pred[i]}  T:{y_true[i]}',
                 color=colour, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle(
    'Adam Model -- Predictions on First 25 Test Images\n'
    '(Green = Correct, Red = Wrong)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

correct_count = np.sum(y_pred == y_true)
print(f"Correct: {correct_count} / 25")


# Step 13: Final Test Evaluation and Summary Table
print("Final evaluation on the held-out test set (10,000 images):")
print("-" * 60)

names  = ['Plain SGD', 'SGD + Momentum', 'Adam']
models = [model_sgd,   model_mom,        model_adam]
hists  = [hist_sgd,    hist_mom,         hist_adam]

for name, model, hist in zip(names, models, hists):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)

    # Find first epoch where val_accuracy crossed 95%
    val_accs = hist.history['val_accuracy']
    epoch_95 = next(
        (i + 1 for i, v in enumerate(val_accs) if v >= 0.95),
        'Not reached in 15 epochs'
    )

    print(f"{name:20s} | "
          f"Test Acc: {acc*100:.2f}%  | "
          f"Epoch to 95%: {epoch_95}")

print("-" * 60)
print("Test set is completely separate from validation set.")
print("These numbers are unbiased estimates of real-world performance.")


# Step 14: Effect of Learning Rate on SGD (Ablation Study)
# Train SGD with three different learning rates to show its sensitivity
lr_values = [0.001, 0.01, 0.1]
lr_histories = {}

for lr in lr_values:
    model_lr = build_model()
    model_lr.compile(
        optimizer=SGD(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    h = model_lr.fit(X_train, y_train,
                     epochs=15, batch_size=128,
                     validation_split=0.1,
                     verbose=0)   # silent training
    lr_histories[lr] = h
    print(f"SGD lr={lr:.3f} | "
          f"Final val accuracy: "
          f"{h.history['val_accuracy'][-1]:.4f}")

# Plot
plt.figure(figsize=(8, 4))
for lr, h in lr_histories.items():
    plt.plot(range(1, 16), h.history['val_accuracy'],
             label=f'SGD lr={lr}', linewidth=2)
plt.title('SGD: Effect of Learning Rate on Val. Accuracy',
          fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# Experiment 4: Activation Functions and Loss Functions - Iris
# ----------------------------------------------------------------------

# Step 1: Install Required Libraries
!pip install numpy pandas matplotlib scikit-learn tensorflow --quiet


# Step 2: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical


# Step 3: Load and Preprocess the Iris CSV
# Load CSV and drop the row-number Id column
df    = pd.read_csv('iris.csv').drop('Id', axis=1)
X_raw = df.iloc[:, :-1].values       # (150, 4)  -- four features
y_raw = df.iloc[:, -1].values        # (150,)    -- species strings

# Encode class labels: setosa=0, versicolor=1, virginica=2
le    = LabelEncoder()
y_int = le.fit_transform(y_raw)      # integer labels
Y     = to_categorical(y_int, 3)     # one-hot (150, 3)

# Normalise features to zero mean and unit variance
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

# Stratified 80/20 train-test split (ensures 10 samples per class in test)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=y_int
)
y_test_int = np.argmax(Y_test, axis=1)   # integer true labels for test set

print("Classes     :", le.classes_)
print("Train shape :", X_train.shape, Y_train.shape)
print("Test  shape :", X_test.shape,  Y_test.shape)


# Step 4: Build the Model Factory Function
def build_model(activation='relu',
                loss='categorical_crossentropy'):
    """
    Returns a freshly compiled model.
    - activation : hidden-layer activation ('relu', 'sigmoid', 'tanh')
    - loss       : loss function  ('categorical_crossentropy' or 'mse')
    """
    tf.random.set_seed(42)
    model = Sequential([
        Dense(16, activation=activation, input_shape=(4,)),
        Dense(8,  activation=activation),
        Dense(3,  activation='softmax')   # output always Softmax for 3-class
    ])
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
    return model


# Step 5: Part A — Compare Activation Functions
activations = ['relu', 'sigmoid', 'tanh']
colours     = ['forestgreen', 'tomato', 'steelblue']
hist_act    = {}

for act in activations:
    model = build_model(activation=act)        # fresh model per activation
    h = model.fit(X_train, Y_train,
                  epochs=150,
                  batch_size=16,
                  validation_data=(X_test, Y_test),
                  verbose=0)
    hist_act[act] = (model, h)
    print(f"{act:8s} -- final val_accuracy: "
          f"{h.history['val_accuracy'][-1]:.4f}")

# Overlay validation accuracy and validation loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for act, col in zip(activations, colours):
    _, h = hist_act[act]
    axes[0].plot(h.history['val_accuracy'],
                 label=act, color=col, linewidth=2)
    axes[1].plot(h.history['val_loss'],
                 label=act, color=col, linewidth=2)

for ax, title, ylabel in zip(
        axes,
        ['Validation Accuracy', 'Validation Loss (CCE)'],
        ['Accuracy', 'Loss']):
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Part A: Activation Function Comparison  '
    '(Loss = CCE, Optimizer = Adam)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


# Step 6: Part B — Compare Loss Functions
loss_configs = {
    'CCE (Categorical CE)': 'categorical_crossentropy',
    'MSE':                  'mse'
}
colours_loss = ['steelblue', 'tomato']
hist_loss    = {}

for label, loss_fn in loss_configs.items():
    model = build_model(activation='relu', loss=loss_fn)
    h = model.fit(X_train, Y_train,
                  epochs=150,
                  batch_size=16,
                  validation_data=(X_test, Y_test),
                  verbose=0)
    hist_loss[label] = (model, h)
    print(f"{label:22s} -- final val_accuracy: "
          f"{h.history['val_accuracy'][-1]:.4f}")

# Plot validation accuracy for both loss functions on the same axis
plt.figure(figsize=(8, 4))
for (label, (_, h)), col in zip(hist_loss.items(), colours_loss):
    plt.plot(h.history['val_accuracy'],
             label=label, color=col, linewidth=2)

plt.title(
    'Part B: Loss Function Comparison\n'
    '(Activation = ReLU, Optimizer = Adam)',
    fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Step 7: Evaluate and Print the Accuracy Summary
print("=" * 55)
print("Part A  --  Activation Function Comparison  (CCE Loss)")
print("=" * 55)
for act in activations:
    model, _ = hist_act[act]
    _, acc   = model.evaluate(X_test, Y_test, verbose=0)
    print(f"  {act:8s}  |  Test Accuracy: {acc * 100:.1f}%")

print()
print("=" * 55)
print("Part B  --  Loss Function Comparison  (ReLU Activation)")
print("=" * 55)
for label, (model, _) in hist_loss.items():
    _, acc = model.evaluate(X_test, Y_test, verbose=0)
    print(f"  {label:22s}  |  Test Accuracy: {acc * 100:.1f}%")

In [ ]:
# ----------------------------------------------------------------------
# Experiment 5: Image Classification using CNN - MNIST
# ----------------------------------------------------------------------

# Step 1: Install Required Libraries
!pip install tensorflow numpy matplotlib --quiet


# Step 2: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D,
                                    Flatten, Dense, Dropout)
from tensorflow.keras.utils import to_categorical


# Step 3: Load and Preprocess MNIST
# Load MNIST -- downloaded and cached automatically by Keras (~11 MB)
(X_train_raw, y_train_raw), \
(X_test_raw,  y_test_raw)  = mnist.load_data()

# Reshape: (n, 28, 28) --> (n, 28, 28, 1)
# Conv2D expects a 4-D tensor: (batch, height, width, channels)
X_train = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32')/ 255.0

# One-hot encode integer labels 0-9 --> 10-dimensional binary vectors
y_train = to_categorical(y_train_raw, num_classes=10)
y_test  = to_categorical(y_test_raw,  num_classes=10)

print("X_train shape :", X_train.shape)
print("X_test  shape :", X_test.shape)
print("y_train shape :", y_train.shape)
print("Pixel range   : [{:.1f}, {:.1f}]".format(
      X_train.min(), X_train.max()))


# Step 4: Build the CNN Model
model = Sequential([
    # --- Convolutional Block 1 ---
    Conv2D(32, (3, 3), activation='relu',
           padding='same', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    # --- Convolutional Block 2 ---
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),

    # --- Classification Head ---
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.summary()


# Step 5: Compile and Train the Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,   # 6000 images held out for validation
    verbose=1
)


# Step 6: Evaluate on the Test Set and Plot Training Curves
# Final evaluation on 10,000 held-out test images
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}  ({test_acc * 100:.2f}%)")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],
             label='Train', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_accuracy'],
             label='Validation', color='darkorange',
             linewidth=2, linestyle='--')
axes[0].set_title('Accuracy over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],
             label='Train', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_loss'],
             label='Validation', color='darkorange',
             linewidth=2, linestyle='--')
axes[1].set_title('Loss over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Step 7: Visualise Predictions on Test Images
# Predict on the first 25 test images
y_pred_prob = model.predict(X_test[:25], verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = np.argmax(y_test[:25], axis=1)

fig, axes = plt.subplots(5, 5, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test_raw[i], cmap='gray')
    ax.axis('off')
    colour = 'green' if y_pred[i] == y_true[i] else 'red'
    ax.set_title(f'P: {y_pred[i]}   T: {y_true[i]}',
                 color=colour, fontsize=9, fontweight='bold')

plt.suptitle(
    'CNN Predictions on First 25 Test Images\n'
    'Green = Correct   |   Red = Wrong',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

correct = int((y_pred == y_true).sum())
print(f"Correct: {correct} / 25")

In [ ]:
# ----------------------------------------------------------------------
# Experiment 6: Transfer Learning with MobileNetV2 - tf_flowers
# ----------------------------------------------------------------------

# Step 1: Install and Import Libraries
!pip install tensorflow-datasets --quiet

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input


# Step 2: Load tf_flowers and Build the tf.data Pipeline
IMG_SIZE    = 96
BATCH_SIZE  = 32
NUM_CLASSES = 5
AUTOTUNE    = tf.data.AUTOTUNE

# Load tf_flowers with a 70/15/15 percentage split
(ds_train, ds_val, ds_test), info = tfds.load(
    'tf_flowers',
    split=['train[:70%]', 'train[70%:85%]', 'train[85%:]'],
    as_supervised=True,
    with_info=True
)

class_names = info.features['label'].names
print("Classes     :", class_names)
print("Total images:", info.splits['train'].num_examples)

# Preprocessing function applied to every image
def preprocess_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = preprocess_input(image)
    return image, label

# Build pipelines
train_ds = (ds_train
            .map(preprocess_image, num_parallel_calls=AUTOTUNE)
            .shuffle(500)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

val_ds = (ds_val
          .map(preprocess_image, num_parallel_calls=AUTOTUNE)
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

test_ds = (ds_test
           .map(preprocess_image, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))


# Step 3: Build the Transfer Learning Model
# Load MobileNetV2 without its top classification layers
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False     # freeze all base layers for Phase 1

# Build model using Functional API
inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = base_model(inputs, training=False)   # keeps BN in inference mode
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()


# Step 4: Phase 1 – Compile and Train the Head
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train for 10 epochs with frozen base
EPOCHS_1 = 10
history1 = model.fit(
    train_ds,
    epochs=EPOCHS_1,
    validation_data=val_ds
)


# Step 5: Phase 2 – Fine-Tune Top Layers of MobileNetV2
# Unfreeze the whole base, then re-freeze early layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Must recompile after changing trainable state
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training for 10 more epochs
EPOCHS_2 = 10
history2 = model.fit(
    train_ds,
    epochs=EPOCHS_1 + EPOCHS_2,
    initial_epoch=EPOCHS_1,
    validation_data=val_ds
)


# Step 6: Plot Training and Validation Curves
# Combine Phase 1 and Phase 2 histories
acc     = history1.history['accuracy']     + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss    = history1.history['loss']         + history2.history['loss']
val_loss= history1.history['val_loss']     + history2.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(acc,     label='Train Acc',  color='steelblue')
axes[0].plot(val_acc, label='Val Acc',    color='orange')
axes[0].axvline(x=9.5, color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy (Phase 1 + Phase 2)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(loss,     label='Train Loss', color='steelblue')
axes[1].plot(val_loss, label='Val Loss',   color='orange')
axes[1].axvline(x=9.5, color='gray', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss (Phase 1 + Phase 2)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()


# Step 7: Evaluate and Visualise Predictions
# Evaluate on test set
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Collect 25 raw images for display
raw_images, raw_labels = [], []
for img, lbl in ds_test.take(25):
    raw_images.append(img.numpy())
    raw_labels.append(lbl.numpy())

# Preprocess those images for prediction
def preprocess_single(img):
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = preprocess_input(img)
    return img

pred_batch = np.array([preprocess_single(img).numpy()
                        for img in raw_images])
preds      = model.predict(pred_batch)        # shape: (25, 5)
pred_labels = np.argmax(preds, axis=1)        # shape: (25,)

# Display 25 predictions in a 5x5 grid
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    display = tf.image.resize(raw_images[i], [96, 96]).numpy().astype('uint8')
    ax.imshow(display)
    true_name = class_names[raw_labels[i]]
    pred_name = class_names[pred_labels[i]]
    colour = 'green' if true_name == pred_name else 'red'
    ax.set_title(f"T:{true_name}\nP:{pred_name}", color=colour, fontsize=7)
    ax.axis('off')
plt.suptitle('Test Predictions  (green = correct, red = wrong)', y=1.01)
plt.tight_layout()
plt.savefig('predictions.png', dpi=150)
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# Experiment 7: Sequence Prediction using LSTM - Airline Passengers
# ----------------------------------------------------------------------

# Step 1: Install Required Libraries
!pip install tensorflow pandas scikit-learn matplotlib --quiet


# Step 2: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error


# Step 3: Load Dataset and Plot Raw Series
# Upload airline-passengers.csv to Colab using the Files panel,
# then load it with Pandas
df = pd.read_csv('airline-passengers.csv')

# Extract the Passengers column as a float32 NumPy array
data = df['Passengers'].values.astype('float32')

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"First 3 rows  :\n{df.head(3)}")
print(f"Value range   : {data.min():.0f} -- {data.max():.0f}(thousands)")

# Plot the raw time series
plt.figure(figsize=(10, 4))
plt.plot(data, color='steelblue', linewidth=1.5)
plt.title('International Airline Passengers (Jan 1949 -- Dec 1960)',
          fontweight='bold')
plt.xlabel('Month Index (0 = Jan 1949)')
plt.ylabel('Passengers (thousands)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Step 4: Preprocess: Scale, Create Sequences, Split, and Reshape
# --- 4a. Normalise to [0, 1] ---
scaler      = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data.reshape(-1, 1))

# --- 4b. Create sliding-window sequences ---
LOOK_BACK = 12   # use 12 months of history to predict the next month

def create_sequences(dataset, look_back=12):
    X, y = [], []
    for i in range(len(dataset) - look_back):
        X.append(dataset[i : i + look_back, 0])   # window of look_back values
        y.append(dataset[i + look_back, 0])        # next value to predict
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, LOOK_BACK)
print(f"Total sequences : {len(X)}")

# --- 4c. Train / Test split (preserve temporal order -- no shuffle) ---
train_size  = int(len(X) * 0.80)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]
print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

# --- 4d. Reshape for LSTM: (samples, timesteps, features) ---
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test  = X_test.reshape(X_test.shape[0],  X_test.shape[1], 1)
print(f"X_train reshaped: {X_train.shape}")
print(f"X_test  reshaped: {X_test.shape}")


# Step 5: Build the Stacked LSTM Model
model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(LOOK_BACK, 1)),   # first LSTM
    Dropout(0.2),
    LSTM(32, return_sequences=False),   # second LSTM
    Dropout(0.2),
    Dense(1)                            # output: one predicted value
])

model.summary()


# Step 6: Compile and Train the Model
model.compile(optimizer='adam',
              loss='mean_squared_error')

history = model.fit(X_train, y_train,
                    epochs=50,
                    batch_size=16,
                    validation_data=(X_test, y_test),
                    verbose=1)


# Step 7: Predict, Inverse Transform, and Plot
# --- Generate predictions on train and test sets ---
train_pred = model.predict(X_train, verbose=0)  # shape (105, 1)
test_pred  = model.predict(X_test,  verbose=0)  # shape (27, 1)

# --- Inverse transform: scaled [0,1] -> original passenger counts ---
train_pred_inv  = scaler.inverse_transform(train_pred)
y_train_inv     = scaler.inverse_transform(y_train.reshape(-1, 1))
test_pred_inv   = scaler.inverse_transform(test_pred)
y_test_inv      = scaler.inverse_transform(y_test.reshape(-1, 1))

# --- Build full-length arrays for plotting ---
# Train predictions start at index LOOK_BACK (first 12 months are
# consumed as the first input window)
train_plot = np.empty_like(data)
train_plot[:] = np.nan
train_plot[LOOK_BACK : LOOK_BACK + len(train_pred_inv)] = \
    train_pred_inv[:, 0]

# Test predictions start immediately after the train segment
test_start = LOOK_BACK + len(train_pred_inv)
test_plot = np.empty_like(data)
test_plot[:] = np.nan
test_plot[test_start : test_start + len(test_pred_inv)] = \
    test_pred_inv[:, 0]

# --- Plot ---
plt.figure(figsize=(12, 5))
plt.plot(data,       color='steelblue', linewidth=1.5, label='Actual')
plt.plot(train_plot, color='orange',    linewidth=1.5,
         label='Train Prediction')
plt.plot(test_plot,  color='red',       linewidth=1.5,
         label='Test Prediction')
plt.axvline(x=test_start, color='gray', linestyle='--',
            linewidth=1.0, label='Train/Test boundary')
plt.legend()
plt.title('Airline Passengers: Actual vs LSTM Prediction',
          fontweight='bold')
plt.xlabel('Month Index')
plt.ylabel('Passengers (thousands)')
plt.tight_layout()
plt.savefig('lstm_predictions.png', dpi=150)
plt.show()


# Step 8: Evaluate with RMSE and Plot Loss Curve
# --- RMSE in the original scale (thousands of passengers) ---
train_rmse = math.sqrt(mean_squared_error(y_train_inv,
                                         train_pred_inv))
test_rmse  = math.sqrt(mean_squared_error(y_test_inv,
                                         test_pred_inv))
print(f"Train RMSE : {train_rmse:.2f} thousand passengers")
print(f"Test  RMSE : {test_rmse:.2f} thousand passengers")

# --- Training loss curve ---
plt.figure(figsize=(9, 3))
plt.plot(history.history['loss'],
         label='Train Loss (MSE)', color='steelblue', linewidth=1.5)
plt.plot(history.history['val_loss'],
         label='Val Loss (MSE)', color='darkorange',
         linewidth=1.5, linestyle='--')
plt.title('LSTM Training Loss over 50 Epochs', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error (scaled)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lstm_loss.png', dpi=150)
plt.show()

In [ ]:
# ----------------------------------------------------------------------
# Experiment 8: Autoencoder and Denoising - MNIST
# ----------------------------------------------------------------------

# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models


# Step 2: Load and Preprocess MNIST
# Load MNIST dataset
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

# Normalise pixel values from [0, 255] to [0.0, 1.0]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# Flatten each 28x28 image to a 784-dimensional vector
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

print("Training set shape :", x_train.shape)   # (60000, 784)
print("Test set shape     :", x_test.shape)     # (10000, 784)


# Step 3: Build the Encoder
LATENT_DIM = 32   # size of the bottleneck representation

# Define encoder using Functional API
encoder_input = keras.Input(shape=(784,), name='encoder_input')
x = layers.Dense(128, activation='relu', name='enc_dense1')(encoder_input)
x = layers.Dense(64,  activation='relu', name='enc_dense2')(x)
latent = layers.Dense(LATENT_DIM, activation='relu', name='latent')(x)

encoder = models.Model(encoder_input, latent, name='encoder')
encoder.summary()


# Step 4: Build the Decoder
# Define decoder using Functional API
decoder_input = keras.Input(shape=(LATENT_DIM,), name='decoder_input')
x = layers.Dense(64,  activation='relu',    name='dec_dense1')(decoder_input)
x = layers.Dense(128, activation='relu',    name='dec_dense2')(x)
decoder_output = layers.Dense(784, activation='sigmoid',
                              name='decoder_output')(x)

decoder = models.Model(decoder_input, decoder_output, name='decoder')
decoder.summary()


# Step 5: Assemble and Compile the Full Autoencoder
# Connect encoder output to decoder input
autoencoder_input  = keras.Input(shape=(784,), name='autoencoder_input')
encoded  = encoder(autoencoder_input)
decoded  = decoder(encoded)

autoencoder = models.Model(autoencoder_input, decoded, name='autoencoder')

# Compile
autoencoder.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)

autoencoder.summary()
print("Total trainable parameters:",
      sum(w.numpy().size for w in autoencoder.trainable_weights))


# Step 6: Train the Autoencoder
history = autoencoder.fit(
    x_train, x_train,          # input == target ( reconstruction)
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)


# Step 7: Plot Training and Validation Loss
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train Loss', color='steelblue')
plt.plot(history.history['val_loss'], label='Val Loss', color='orange')
plt.title('Autoencoder Training Loss (Binary Cross-Entropy)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig('ae_loss_curve.png', dpi=150)
plt.show()


# Step 8: Visualise Original vs. Reconstructed Images
# Generate reconstructions for the first 10 test images
reconstructed = autoencoder.predict(x_test[:10])

n = 10
fig, axes = plt.subplots(2, n, figsize=(15, 3))

for i in range(n):
    # Top row: original images
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=9)

    # Bottom row: reconstructed images
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstructed', fontsize=9)

plt.suptitle('Autoencoder -- Original vs. Reconstructed', y=1.02)
plt.tight_layout()
plt.savefig('ae_reconstructions.png', dpi=150)
plt.show()


# Step 9: Denoising Application
# Add Gaussian noise to test images
noise_factor = 0.4
x_test_noisy = x_test + noise_factor * np.random.normal(
                    loc=0.0, scale=1.0, size=x_test.shape)
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0)

# Reconstruct (denoise) using the trained autoencoder
x_test_denoised = autoencoder.predict(x_test_noisy)

# Display: original | noisy | denoised
n = 10
fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
row_labels = ['Original', 'Noisy', 'Denoised']

for i in range(n):
    for row, img in enumerate([x_test[i], x_test_noisy[i],
                               x_test_denoised[i]]):
        axes[row, i].imshow(img.reshape(28, 28), cmap='gray')
        axes[row, i].axis('off')
        if i == 0:
            axes[row, i].set_ylabel(row_labels[row], fontsize=9,
                                   rotation=90, labelpad=4)

plt.suptitle('Denoising Autoencoder -- Original vs. Noisy vs.Denoised',
             y=1.02)
plt.tight_layout()
plt.savefig('ae_denoising.png', dpi=150)
plt.show()